# PDEForge for Uncertainty Quantification

This notebook demonstrates how to use PDEForge to generate datasets specifically designed for **uncertainty quantification (UQ)** in neural operators.

## Key Concepts

1. **Calibration Split**: A held-out dataset for calibrating prediction intervals
2. **Parameter Exploration**: Understanding how inputs affect outputs
3. **Integration with Operator_UQ**: End-to-end workflow

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, '..')

from pdeforge import generate_dataset, list_models, describe_model

## 1. Why UQ Matters for Operator Learning

Neural operators learn mappings between function spaces:

$$\mathcal{G}_\theta: \mathcal{A} \rightarrow \mathcal{U}$$

But they don't tell us **how confident** we should be in their predictions.

**UQ methods** (like conformal prediction) provide:
- Prediction intervals with coverage guarantees
- Calibrated uncertainty estimates
- Detection of out-of-distribution inputs

**The catch**: UQ methods require a **calibration set** that is:
- Separate from training data
- Separate from validation data (used for hyperparameter tuning)
- Representative of the test distribution

## 2. The Four-Way Split

PDEForge provides a dedicated calibration split:

In [ ]:
# Generate a Burgers dataset
dataset = generate_dataset(
    model="burgers_1d",
    n_samples=1000,
    resolution={"x": 256},
    params={"viscosity": 0.01},
    seed=42,
)

# Split with dedicated calibration set
splits = dataset.split(
    train=0.60,  # 600 samples for training
    val=0.15,    # 150 samples for validation
    cal=0.15,    # 150 samples for calibration ← For UQ!
    test=0.10,   # 100 samples for final evaluation
    seed=42,
)

print("Dataset splits for UQ workflow:")
print("-" * 40)
for name, ds in splits.items():
    print(f"{name:8s}: {ds.n_samples:4d} samples")

### Why a Separate Calibration Set?

| Split | Purpose | Used By |
|-------|---------|--------|
| **train** | Learn model parameters | Optimizer |
| **val** | Tune hyperparameters | Early stopping, LR scheduling |
| **cal** | Calibrate uncertainty | Conformal prediction, Platt scaling |
| **test** | Final evaluation | You (once!) |

Using validation data for calibration would **overfit** the uncertainty estimates!

## 3. Typical UQ Workflow

Here's how PDEForge integrates with Operator_UQ:

In [ ]:
# This is a conceptual workflow - Operator_UQ integration coming soon!

workflow = """
# Step 1: Generate data with PDEForge
from pdeforge import generate_dataset

dataset = generate_dataset("burgers_1d", n_samples=10000, ...)
splits = dataset.split(train=0.6, val=0.15, cal=0.15, test=0.1)

# Step 2: Train neural operator with Operator_UQ
from operator_uq import FNO, MCDropout

model = FNO(modes=16, width=64, dropout=0.1)
model.fit(
    splits['train'].inputs, 
    splits['train'].outputs,
    val_data=(splits['val'].inputs, splits['val'].outputs),
)

# Step 3: Calibrate uncertainty estimates
from operator_uq import ConformalPredictor

# Use calibration set (NOT validation set!)
predictor = ConformalPredictor(model)
predictor.calibrate(
    splits['cal'].inputs,
    splits['cal'].outputs,
)

# Step 4: Make predictions with guaranteed coverage
y_pred, intervals = predictor.predict(
    splits['test'].inputs,
    confidence=0.9,  # 90% coverage
)

# Check coverage on test set
coverage = compute_coverage(splits['test'].outputs, intervals)
print(f"Empirical coverage: {coverage:.1%}")  # Should be ~90%
"""

print(workflow)

## 4. Understanding Your Data: Parameter Exploration

Before training, it's valuable to understand how physical parameters affect solutions.

PDEForge provides exploration utilities:

In [ ]:
# First, understand what parameters are available
print(describe_model("burgers_1d"))

In [ ]:
from pdeforge import explore_parameter, visualize_parameter_effect

# Explore how viscosity affects the solution
viscosity_study = explore_parameter(
    model="burgers_1d",
    param_name="viscosity",
    param_values=[0.001, 0.01, 0.05, 0.1],
    resolution={"x": 256},
    n_samples_per_value=3,
    seed=42,
)

print(f"Generated {viscosity_study.n_samples} samples across 4 viscosity values")

In [ ]:
# Visualize the effect
visualize_parameter_effect(viscosity_study)

### Interpretation

- **Low viscosity** (ν = 0.001): Sharp shocks, discontinuous solutions
- **High viscosity** (ν = 0.1): Smooth, diffused solutions

This affects UQ:
- Sharp shocks are harder to predict → higher uncertainty
- Smooth solutions are easier → lower uncertainty

Your model should capture this!

## 5. Generating Data for Different UQ Scenarios

### Scenario A: In-Distribution UQ

Train and test on the same parameter regime:

In [ ]:
# Same viscosity for train and test
in_dist_data = generate_dataset(
    model="burgers_1d",
    n_samples=2000,
    resolution={"x": 256},
    params={"viscosity": 0.01},
    seed=42,
)

splits_in_dist = in_dist_data.split(train=0.6, val=0.15, cal=0.15, test=0.1)

### Scenario B: Out-of-Distribution Detection

Train on one regime, test on another:

In [ ]:
# Training data: moderate viscosity
train_data = generate_dataset(
    model="burgers_1d",
    n_samples=1500,
    resolution={"x": 256},
    params={"viscosity": 0.01},
    seed=42,
)

# OOD test data: very low viscosity (sharper shocks)
ood_test_data = generate_dataset(
    model="burgers_1d",
    n_samples=200,
    resolution={"x": 256},
    params={"viscosity": 0.001},  # Different!
    seed=123,
)

print("Training viscosity: 0.01")
print("OOD test viscosity: 0.001")
print("\nA good UQ method should show higher uncertainty on OOD data!")

### Scenario C: Mixed Training for Robustness

Train on multiple parameter regimes:

In [ ]:
from pdeforge import explore_parameter_grid

# Generate data across a grid of parameters
mixed_data = explore_parameter_grid(
    model="burgers_1d",
    param_grid={
        "viscosity": [0.005, 0.01, 0.02, 0.05],
    },
    resolution={"x": 256},
    n_samples_per_config=250,
    seed=42,
)

print(f"Mixed training data: {mixed_data.n_samples} samples")
print(f"Covers viscosity range: 0.005 to 0.05")

## 6. Saving Data for Operator_UQ

Save your datasets in a format ready for training:

In [ ]:
# Save the full dataset
dataset.save("./burgers_uq_data")

# Or save splits separately
for name, split_ds in splits.items():
    split_ds.save(f"./burgers_uq_data/{name}")
    
print("Saved dataset with splits for UQ workflow")

In [ ]:
# Clean up
import shutil
shutil.rmtree("./burgers_uq_data")

## Summary

PDEForge supports UQ workflows through:

1. **Calibration splits**: Dedicated `cal` split for calibrating uncertainty
2. **Parameter exploration**: Understand how physics affects predictions
3. **Flexible generation**: Create in-distribution, OOD, and mixed datasets
4. **Easy integration**: Save in formats compatible with Operator_UQ

### Next Steps

- See `04_stochastic_models.ipynb` for stochastic PDE datasets (coming soon)
- Check out [Operator_UQ](https://github.com/your-org/Operator_UQ) for the full UQ pipeline